# Claude Study 03 — Configuring Claude Code

**Companion notebook to** `Claude_Study_03_Configuring_Claude_Code.md`

This notebook does something the first two could not: it **verifies claims against the Claude Code build itself**, not just the API. Four of the course's statements about settings turn out to be wrong or incomplete, and every one of them is checkable in under a minute.

### How to use this
Each section opens with three lines:
- **Concept.** which Part of the study doc it belongs to
- **Surface.** the exact file, key or command involved
- **What this proves.** the claim being tested

### Two ways to run it
| | Works for | Notes |
|---|---|---|
| **Locally, with Claude Code installed** | §1–§7 (everything) | the binary-introspection cells need the CLI on disk |
| **Colab** | §0, §3, §8 | no Claude Code binary in Colab; those cells are marked and will skip cleanly |

⚠️ **Nothing here modifies your real configuration.** Every write goes to a throwaway temp directory. The one cell that *reads* `~/.claude/settings.json` redacts values before printing.

## §0 — Setup

**Concept.** Prerequisite, not a lesson section.
**Surface.** The Claude Code binary on disk, plus Python stdlib only.
**What this proves.** That we have a real build to introspect — and which version, because every claim below is version-specific.

No API key needed for most of this notebook. The settings schema lives inside the shipped binary.

In [ ]:
import os, re, json, shutil, subprocess, tempfile, pathlib, sys

def find_claude_binary():
    """Locate the Claude Code executable. Returns (path, version) or (None, None)."""
    # Preferred: the versioned binaries directory
    vdir = pathlib.Path.home() / ".local/share/claude/versions"
    if vdir.is_dir():
        vers = sorted(
            [p for p in vdir.iterdir() if p.is_file() and os.access(p, os.X_OK)],
            key=lambda p: [int(x) for x in re.findall(r"\d+", p.name)] or [0],
        )
        if vers:
            return vers[-1], vers[-1].name
    # Fallback: whatever `claude` resolves to
    w = shutil.which("claude")
    if w:
        real = pathlib.Path(os.path.realpath(w))
        return real, real.name
    return None, None

BIN, VERSION = find_claude_binary()
HAVE_BIN = BIN is not None and BIN.is_file()

if HAVE_BIN:
    size_mb = BIN.stat().st_size / 1e6
    print(f"Claude Code binary : {BIN}")
    print(f"Version            : {VERSION}")
    print(f"Size               : {size_mb:,.0f} MB")
else:
    print("No Claude Code binary found (expected in Colab).")
    print("Sections that introspect the build will skip; §0, §3 and §8 still run.")

### §0b — Load the binary once

**Concept.** Part 4 — the settings catalog.
**Surface.** The shipped executable; the settings schema is embedded in it as Zod builders.
**What this proves.** That the authoritative key list is *on your machine already* — you never have to trust a slide.

The binary is a few hundred MB. We decode it once, tolerating invalid bytes, and reuse the string.

In [ ]:
BLOB = ""
if HAVE_BIN:
    BLOB = BIN.read_bytes().decode("utf-8", "replace")
    print(f"Decoded {len(BLOB):,} characters.")
else:
    print("Skipped — no binary.")

## §1 — The settings catalog, extracted from the build

**Concept.** Part 4 — all 145 settings keys.
**Surface.** Zod schema builders of the form `key:()=>…​.describe("…")`.
**What this proves.** The complete, current settings surface — including keys no slide mentions.

**The gotcha that cost me a wrong answer the first time.** Two of them, actually, and both are worth seeing:

1. Keys are defined as `key:()=>…`, never as the quoted string `"key"`. A naive `grep '"spinnerVerbs"'` reports **ABSENT** for a key that exists.
2. Descriptions use **both** quote styles. A regex that only handles `"` will skip a single-quoted description and silently attach the *next* key's text to this key — which is exactly how I first "discovered" that `pluginTrustMessage` meant *"Color theme for the UI"*. It doesn't.

So the pattern below does two defensive things: it accepts either quote character, and it **refuses to match across another `()=>`**, which is what stops the bleed.

In [ ]:
SCHEMA = {}
if HAVE_BIN:
    pat = re.compile(
        r"([A-Za-z_][A-Za-z0-9_]*):\(\)=>"      # the key
        r"((?:(?!\(\)=>).){0,900}?)"            # body — must not cross another ()=>
        r"\.describe\(\s*([\"'])((?:(?!\3).){0,1200})\3\s*\)",
        re.S,
    )
    for m in pat.finditer(BLOB):
        k, desc = m.group(1), m.group(4)
        if k not in SCHEMA:
            SCHEMA[k] = re.sub(r"\s+", " ", desc.replace("\\n", " ").replace('\\"', '"').replace("\\'", "'"))
    print(f"Extracted {len(SCHEMA)} settings keys with descriptions.\n")
    for k in sorted(SCHEMA)[:12]:
        print(f"  {k}")
    print(f"  … and {len(SCHEMA)-12} more")
else:
    print("Skipped — no binary.")

# ---- Fallback for environments without the binary (Colab) -------------------
# Prefer LIVE extraction above. If there was no binary, load the committed
# snapshot so the rest of the notebook still runs. Provenance is printed so you
# always know which you are looking at.
if not SCHEMA:
    import urllib.request
    SNAP = ("https://raw.githubusercontent.com/sunilmogadati/production-ai-engineering/"
            "main/data/claude_code_settings_schema.json")
    try:
        local = pathlib.Path("../data/claude_code_settings_schema.json")
        raw = local.read_text() if local.exists() else urllib.request.urlopen(SNAP, timeout=30).read().decode()
        snap = json.loads(raw)
        SCHEMA = snap["keys"]
        EVIDENCE = snap.get("evidence", {})
        pv = snap["_provenance"]
        print("SOURCE: committed SNAPSHOT (no local binary found)")
        print(f"  extracted from build {pv['version']} on {pv['extracted']}")
        print(f"  {len(SCHEMA)} keys\n")
        print("  ⚠️  A snapshot can go stale. With Claude Code installed locally,")
        print("      the cell above extracts from YOUR build instead.")
    except Exception as e:
        print("Snapshot unavailable:", e)
else:
    EVIDENCE = {}
    print(f"SOURCE: LIVE extraction from build {VERSION} on this machine.")


### §1b — The adversarial arm: prove the bleed-guard matters

**Concept.** Part 4 — verification method, not a settings fact.
**Surface.** The same regex, with the `(?!\(\)=>)` guard removed.
**What this proves.** That the guard is load-bearing — without it, descriptions attach to the wrong keys.

If removing the guard changed nothing, my "fix" was superstition. Let's check rather than assume.

In [ ]:
if HAVE_BIN and BLOB:
    sloppy = re.compile(
        r"([A-Za-z_][A-Za-z0-9_]*):\(\)=>(.{0,600}?)\.describe\(\"((?:[^\"\\\\]|\\\\.){0,400})\"\)",
        re.S,
    )
    SLOPPY = {}
    for m in sloppy.finditer(BLOB):
        k, d = m.group(1), m.group(3)
        if k not in SLOPPY:
            SLOPPY[k] = re.sub(r"\s+", " ", d)

    probe = "pluginTrustMessage"
    print(f"key: {probe}\n")
    print(f"  sloppy regex : {SLOPPY.get(probe, '(not found)')[:110]}")
    print(f"  guarded regex: {SCHEMA.get(probe, '(not found)')[:110]}")

    differing = [k for k in SCHEMA if k in SLOPPY and SCHEMA[k] != SLOPPY[k]]
    print(f"\nKeys where the two regexes disagree: {len(differing)}")
    print("→ the guard is load-bearing" if differing else "→ guard made no difference")
else:
    print("Skipped — this check needs the raw binary; it cannot run on the snapshot.")

## §2 — Correction 1: `cleanupPeriodDays: 0` is rejected

**Concept.** Part 4 — session and storage settings.
**Surface.** `cleanupPeriodDays` in `settings.json`.
**What this proves.** The course teaches removed behaviour. The slide says *"Set to 0 to delete ALL transcripts at startup and disable persistence entirely."* In this build, `0` is refused.

The error text the build ships is unusually candid about *why* it changed — read it in full.

In [ ]:
if HAVE_BIN and BLOB:
    m = re.search(r"cleanupPeriodDays must be at least.{0,420}", BLOB, re.S)
    err = m.group(0).split("\x00")[0] if m else None
    src = "LIVE from this machine's binary"
else:
    err = EVIDENCE.get("cleanupPeriodDays_error")
    src = "committed snapshot"

print(f"ERROR STRING SHIPPED IN THE BUILD  ({src}):\n")
print(err or "(not found)")
print("\n" + "-"*66)
print("Schema description:\n ", SCHEMA.get("cleanupPeriodDays", "(not found)"))

**Why this is the best teaching artifact in the whole lesson.**

`0` was ambiguous between two opposite intentions — *"never clean up"* and *"keep nothing"* — and it silently resolved to the **destructive** one. People setting it to mean "retain forever" lost every transcript.

The fix wasn't to pick a winner. It was to **reject the ambiguous value** and force the caller to say which they meant:

| Intent | Correct expression |
|---|---|
| keep transcripts ~forever | `"cleanupPeriodDays": 3650` |
| write no transcripts at all | remove the setting; use `--no-session-persistence` (needs `--print`) or SDK `persistSession: false` |
| `0` | rejected |

That is a design lesson worth more than the setting itself: when one value means two opposite things to different users, the safe move is to make it an error.

## §3 — Correction 2: settings precedence runs the other way

**Concept.** Part 2 — the precedence ladder.
**Surface.** `.claude/settings.json` vs `.claude/settings.local.json`, same key.
**What this proves.** **Local overrides project.** The course's scope diagram reads Managed → User → Project → Local top-to-bottom; read as a precedence ladder it implies User beats Project beats Local. The build does the opposite.

This section runs **anywhere Claude Code is installed** and touches nothing outside a temp directory.

First, the static evidence from the build — the source list in ascending precedence order:

In [ ]:
if HAVE_BIN and BLOB:
    from collections import Counter
    rows = Counter(re.findall(r'\[(?:\"(?:user|project|local|flag|policy)Settings\",?\s*){2,}\]', BLOB)).most_common(5)
    rows = [(a, n) for a, n in rows]
    src = "LIVE"
else:
    rows = [(e["array"], e["occurrences"]) for e in EVIDENCE.get("settings_source_orders", [])][:5]
    src = "snapshot"

print(f"source-order arrays found in the build ({src}):\n")
for arr, n in rows:
    print(f"{n:>4}x  {arr}")
print("\n→ ascending: user < project < local < flag(--settings) < policy(managed)")

### §3b — Now execute it

Two settings files, same key, nothing else different. We set an environment variable, then ask a throwaway session to print it.

**Cost:** one tiny Haiku call. Pennies.

In [ ]:
import tempfile, subprocess, json, os, pathlib

def precedence_experiment(model="claude-haiku-4-5"):
    d = pathlib.Path(tempfile.mkdtemp(prefix="prec_"))
    (d / ".claude").mkdir()
    (d / ".claude/settings.json").write_text(json.dumps({"env": {"PRECTEST": "PROJECT_WINS"}}))
    (d / ".claude/settings.local.json").write_text(json.dumps({"env": {"PRECTEST": "LOCAL_WINS"}}))
    print(f"scratch dir: {d}")
    print("  .claude/settings.json       -> PRECTEST=PROJECT_WINS")
    print("  .claude/settings.local.json -> PRECTEST=LOCAL_WINS\n")
    r = subprocess.run(
        ["claude", "-p", "Run the bash command: env | grep PRECTEST   Then reply with only the line it printed.",
         "--model", model, "--allowedTools", "Bash", "--permission-mode", "acceptEdits"],
        cwd=d, capture_output=True, text=True, timeout=300,
    )
    return r.stdout.strip().splitlines()[-1] if r.stdout.strip() else f"(no output) {r.stderr[-300:]}"

if shutil.which("claude"):
    out = precedence_experiment()
    print("RESULT:", out)
    print("\n→ LOCAL wins." if "LOCAL" in out else "\n→ unexpected — investigate")
else:
    print("Skipped — `claude` not on PATH (expected in Colab).")

**The consequence to teach.** A team lead who puts a setting in the committed `.claude/settings.json` has **not** locked it down. Any developer overrides it in their own `settings.local.json`, which is gitignored, so the override is invisible in review.

Only **managed settings** are un-overridable — which is the entire reason Part 6's lockdown flags exist.

## §4 — Correction 3: Default *can* be constrained

**Concept.** Part 4 — model settings.
**Surface.** `availableModels`, `enforceAvailableModels`, `availableModelsMatch`.
**What this proves.** The slide's *"Does NOT affect the 'Default' option in the model picker"* is true of `availableModels` alone — and leaves an administrator believing Default cannot be locked. There is a companion key that locks it.

In [ ]:
if SCHEMA:
    for k in ["availableModels", "enforceAvailableModels", "availableModelsMatch", "modelOverrides"]:
        print(f"\n### {k}\n{SCHEMA.get(k, '(not found)')}")
else:
    print("Skipped — no schema.")

Note `availableModelsMatch` too: the default is `"prefix"`, so an entry of `"claude-opus-5"` **also allows `claude-opus-5-5"`**. If you meant to pin exactly one version, that is `"exact"`.

This connects straight back to Study 02: the prompt-caching minimum is **model-dependent and non-monotonic** (512 → 4,096 tokens). A prefix match that quietly admits a different model version can therefore change your cache economics without anyone editing a prompt.

## §5 — Correction 4: CLAUDE.md imports stop silently

**Concept.** Part 3 — imports.
**Surface.** `@path` imports in `CLAUDE.md`.
**What this proves.** "Maximum depth of five hops" means **the root plus four levels of imports**. The fifth level is dropped — with no warning.

We build a chain where each file carries a unique token, then ask a session which tokens it can see. Tools are disabled so it cannot cheat by reading the files.

In [ ]:
def build_chain(depth=7):
    d = pathlib.Path(tempfile.mkdtemp(prefix="hop_"))
    (d / "CLAUDE.md").write_text("# Hop test\n\nTOKEN_L0 is ALPHA0.\n\n@a1.md\n")
    for i in range(1, depth + 1):
        body = f"TOKEN_L{i} is ALPHA{i}.\n"
        if i < depth:
            body += f"\n@a{i+1}.md\n"
        (d / f"a{i}.md").write_text(body)
    return d

def ask_visible_tokens(d, model="claude-haiku-4-5"):
    r = subprocess.run(
        ["claude", "-p",
         "List every TOKEN_Ln value you can see in your loaded context, as a bare comma-separated "
         "list like ALPHA0,ALPHA1. Do not read any files with tools. If you see none say NONE.",
         "--model", model, "--permission-mode", "manual",
         "--disallowed-tools", "Read,Bash,Glob,Grep,Edit,Write,WebFetch,WebSearch,Task"],
        cwd=d, capture_output=True, text=True, timeout=300,
    )
    return r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "(no output)"

if shutil.which("claude"):
    d = build_chain(7)
    print(f"chain: CLAUDE.md -> a1 -> ... -> a7   ({d})\n")
    res = ask_visible_tokens(d)
    print("VISIBLE:", res)
    seen = sorted(int(x) for x in re.findall(r"ALPHA(\d+)", res))
    print(f"\nlevels loaded: {seen}")
    print(f"→ root + {max(seen)} levels of imports" if seen else "→ nothing loaded")
else:
    print("Skipped — `claude` not on PATH.")

### §5b — The adversarial arm

`a5.md` didn't load. There is a boring alternative explanation: the file is broken, or that token is somehow special.

So: **the same `a5.md`, imported directly from the root.** If it appears, the cutoff is genuinely depth. If it still doesn't, my conclusion was wrong and the problem is the file.

In [ ]:
if shutil.which("claude"):
    d2 = pathlib.Path(tempfile.mkdtemp(prefix="hopctl_"))
    (d2 / "CLAUDE.md").write_text("# Control\n\nTOKEN_L0 is ALPHA0.\n\n@a5.md\n")
    (d2 / "a5.md").write_text("TOKEN_L5 is ALPHA5.\n")   # identical content, depth 1
    print("control: CLAUDE.md -> a5.md  (depth 1)\n")
    res2 = ask_visible_tokens(d2)
    print("VISIBLE:", res2)
    print("\n→ loads fine at depth 1, so the cutoff is DEPTH, not the file."
          if "5" in res2 else "\n→ still missing — the file itself is the problem, not depth.")
else:
    print("Skipped — `claude` not on PATH.")

**The dangerous word is *silently*.** No warning, no error, no entry in `/doctor`. A deep import tree loses its leaves, and the only symptom is Claude behaving as though a convention you carefully documented does not exist.

Keep import trees shallow, and verify the leaves actually arrived — this cell is how.

## §6 — Correction 5: `attribution` replaces `includeCoAuthoredBy`

**Concept.** Part 4 — git attribution settings.
**Surface.** `includeCoAuthoredBy` (deprecated) → `attribution` (string).
**What this proves.** The replacement is strictly better: a boolean became a string you control, trailers included.

The course mentions the deprecation but not what replaced it.

In [ ]:
if SCHEMA:
    for k in ["includeCoAuthoredBy", "attribution", "includeGitInstructions"]:
        print(f"\n### {k}\n{SCHEMA.get(k, '(not found)')}")
else:
    print("Skipped — no schema.")

### Why this one is immediately useful

If you have a standing convention like *"never add `Co-Authored-By` trailers to my commits,"* writing it in `CLAUDE.md` makes it a **request** — Part 0's whole point. One line of configuration makes it a **guarantee**:

```jsonc
// ~/.claude/settings.json
{ "attribution": "" }
```

Empty string hides attribution entirely. That is Part 0 applied: move anything that must not fail out of context and into configuration.

In [ ]:
# Read-only peek at your real user settings — VALUES ARE REDACTED.
user_settings = pathlib.Path.home() / ".claude/settings.json"
if user_settings.exists():
    try:
        cfg = json.loads(user_settings.read_text())
        print(f"{user_settings}\n")
        for k in sorted(cfg):
            v = cfg[k]
            shape = type(v).__name__ if not isinstance(v, (str, int, bool)) else "scalar"
            print(f"  {k:<34} [{shape}]")
        print("\nattribution set?      ", "attribution" in cfg)
        print("includeCoAuthoredBy?  ", "includeCoAuthoredBy" in cfg, "(deprecated if present)")
    except Exception as e:
        print("could not parse:", e)
else:
    print("No ~/.claude/settings.json on this machine.")

## §7 — Browse the catalog by category

**Concept.** Part 4 — the whole settings surface.
**Surface.** The extracted schema.
**What this proves.** Nothing new — this is the reference cell. Run it when you need to find which key does a thing.

The course's advice is right: don't memorise these. Know the *categories*, then look it up.

In [ ]:
CATEGORIES = {
    "Authentication":  ["apiKeyHelper","awsAuthRefresh","awsCredentialExport","forceLoginMethod","forceLoginOrgUUID"],
    "Session/storage": ["cleanupPeriodDays","autoMemoryDirectory","plansDirectory"],
    "Model":           ["model","availableModels","availableModelsMatch","enforceAvailableModels",
                        "modelOverrides","alwaysThinkingEnabled","fastModePerSessionOptIn"],
    "Output/language": ["outputStyle","language","includeGitInstructions"],
    "Git attribution": ["attribution","includeCoAuthoredBy"],
    "Announcements":   ["companyAnnouncements","autoUpdatesChannel"],
    "Spinner/UI":      ["showTurnDuration","spinnerVerbs","spinnerTipsEnabled","spinnerTipsOverride",
                        "terminalProgressBarEnabled","prefersReducedMotion"],
    "Agent teams":     ["teammateMode"],
    "MCP":             ["enableAllProjectMcpServers","enabledMcpjsonServers","disabledMcpjsonServers",
                        "allowedMcpServers","deniedMcpServers"],
    "Lockdown (managed only)": ["allowManagedPermissionRulesOnly","allowManagedMcpServersOnly",
                        "allowManagedHooksOnly","strictKnownMarketplaces","blockedMarketplaces","pluginTrustMessage"],
    "Hooks":           ["disableAllHooks","allowedHttpHookUrls","httpHookAllowedEnvVars"],
    "Observability":   ["otelHeadersHelper","statusLine","fileSuggestion","respectGitignore"],
}

if SCHEMA:
    listed = set()
    for cat, keys in CATEGORIES.items():
        print(f"\n{'='*70}\n{cat}\n{'='*70}")
        for k in keys:
            listed.add(k)
            d = SCHEMA.get(k)
            print(f"\n  {k}\n    {d[:160] if d else '(not in this build)'}")
    rest = sorted(set(SCHEMA) - listed)
    print(f"\n\n{len(rest)} further keys exist in this build, uncategorised above:\n")
    print("  " + ", ".join(rest[:40]) + (" …" if len(rest) > 40 else ""))
else:
    print("Skipped — no schema (run §1 with a local binary).")

## §8 — Permission rules: a dry-run you can reason about

**Concept.** Part 5 — `allow` / `ask` / `deny`.
**Surface.** `permissions` in `settings.json`.
**What this proves.** The evaluation order — **deny wins, then ask, then allow, then `defaultMode`**.

This cell is a **model** of the documented order, not a probe of the binary. It is here so a cohort can reason about rule interactions without burning API calls — and it is labelled as a model precisely because Study 01 taught us not to present unexecuted logic as measured behaviour.

Runs anywhere, including Colab.

In [ ]:
import fnmatch

PERMISSIONS = {
    "allow": ["Bash(git diff *)", "Bash(git log *)", "Bash(npm run *)", "Read(**)", "Write(./src/**)"],
    "ask":   ["Bash(git commit *)", "Bash(git push *)"],
    "deny":  ["WebFetch", "Bash(curl *)", "Bash(wget *)", "Bash(rm -rf *)",
              "Read(./.env)", "Read(./secrets/**)", "Read(~/.ssh/**)"],
    "defaultMode": "acceptEdits",
}

def matches(rule, request):
    if "(" not in rule:
        return request.split("(")[0] == rule          # bare tool name, e.g. "WebFetch"
    return fnmatch.fnmatch(request, rule)

def decide(request, perms=PERMISSIONS):
    """MODEL of the documented order: deny > ask > allow > defaultMode."""
    for bucket, verdict in (("deny", "BLOCKED"), ("ask", "PROMPT"), ("allow", "ALLOWED")):
        for rule in perms.get(bucket, []):
            if matches(rule, request):
                return verdict, rule
    return f"defaultMode={perms.get('defaultMode')}", "(no rule matched)"

tests = [
    "Bash(git diff HEAD)", "Bash(git commit -m 'x')", "Bash(rm -rf /tmp/x)",
    "Read(./.env)", "Read(./src/app.py)", "Write(./src/new.py)",
    "Write(./infra/main.tf)", "WebFetch(https://example.com)",
]
print(f"{'REQUEST':<34} {'VERDICT':<22} MATCHED RULE")
print("-"*92)
for t in tests:
    v, r = decide(t)
    print(f"{t:<34} {v:<22} {r}")

Two things to pull out for a class:

- **`Read(./.env)` is `BLOCKED`.** Reading a credential is exfiltration, not a benign operation. The three `Read` deny rules are the most important lines in that config.
- **`Write(./infra/main.tf)` matches nothing** and falls through to `defaultMode: "acceptEdits"` — so it is *accepted silently*. Allowlists leave gaps by construction; the `deny` list and the `defaultMode` are what govern the gap.

And the Study 01 callback: this is the same distinction as **`tools` vs `allowed_tools`** in the Agent SDK. Availability and pre-approval are different axes. `deny` is the only one that is a wall.

## Where this lands

| Course claim | Reality | Section |
|---|---|---|
| `cleanupPeriodDays: 0` disables persistence | **rejected**, minimum 1 | §2 |
| scope diagram implies User > Project > Local | **Local > Project > User** | §3 |
| `availableModels` can't constrain Default | **`enforceAvailableModels`** does | §4 |
| imports go "five hops" | **root + 4 levels**, dropped silently | §5 |
| `includeCoAuthoredBy` deprecated | → **`attribution`** (string, `""` hides) | §6 |

**The method, one more time:** every correction above came from either introspecting the shipped build or executing the behaviour — never from recalling how it used to work. Two of them (§3, §5) needed a second, adversarial cell before the result meant anything, and §1b exists only to prove my own parsing wasn't lying to me.

**The idea worth keeping** is Part 0: `settings.json` is **enforced**, `CLAUDE.md` is **persuasion**. When a rule must not fail, it belongs on the left.